# Algorithm Documentation — 10 Benchmarked Algorithms

This notebook documents the 10 algorithms selected for the final benchmark in `entrega_final.ipynb`, after the candidate screening phase (27 → 10).

Algorithms are grouped into:

1. **Baseline algorithms** — standard KNN, frequency-weighted KNN, and SMOTE
2. **FairRank family** — the main contribution: rank-correction for class imbalance

For each algorithm the document covers:
- **Theory** — mathematical derivation and the problem it addresses
- **How it works** — step-by-step algorithmic walkthrough
- **Hyperparameters** — configurable parameters and their effects
- **Time complexity** — asymptotic cost at training and inference time

---

## Contents

**Baseline Algorithms**

- 1.1 [KNNOptK](#knnoptk) — Standard KNN with cross-validated $k$
- 1.2 [KNNWeighted](#knnweighted) — Class-frequency vote weighting
- 1.3 [SMOTE+KNN](#smote-knn) — Synthetic oversampling + standard KNN

**FairRank Family**

- [2. FairRank Family](#fairrank-family) — Poisson model → dimension-free fair rank derivation
- 2.1 [KNNFairRank](#knnfairrank) — Core rank-corrected voting algorithm
- 2.2 [KNNFairRankCV](#knnfairrankcv) — Adaptive correction strength via inner CV on $\alpha$
- 2.3 [KNNFairRankJointCV](#knnfairrankjointcv) — Joint CV over $(\alpha,\ n_\text{votes})$ grid
- 2.4 [KNNFairRankJackknife](#knnfairrankjackknife) — LOO minority robustness at inference time
- 2.5 [KNNFairRankOptVotes](#knnfairrankoptvotes) — Adaptive vote count via inner CV on $n_\text{votes}$
- 2.6 [KNNFairRankEnsemble](#knnfairrankensemble) — Full-grid vote pooling at inference time
- 2.7 [KNNFairRankTopoJointBootstrap](#knnfairranktopo) — Topology-adaptive per-region $k_\text{eff}$ with OOB gating

---

<a id="problem"></a>

## 0. The Class-Imbalance Problem

Standard KNN votes by majority among the $k$ nearest neighbours. When the training set has far more majority-class points than minority-class points (imbalance ratio $r = N_{maj} / N_{min} \gg 1$), two things go wrong:

1. **Vote dilution.** Even near the true decision boundary, the $k$ neighbours are almost certainly dominated by majority points — just because there are more of them.
2. **Sampling bias in distances.** The expected minimum distance to the majority class is smaller than to the minority class, even when both classes have identical underlying densities. This is a pure statistical artefact from having more samples.

All algorithms in this project address one or both of these problems.

**Key tension in evaluation:** G-Mean rewards algorithms that identify minority cases aggressively (high recall). MCC and F1 also penalise false positives, so they reward precision. An algorithm that predicts minority too eagerly wins G-Mean but loses MCC. This trade-off shapes the design of every variant in the FairRank family.

---

<a id="baselines"></a>

## 1. Baseline Algorithms

### Base class: KNNClassifier & KNNClassifierFast

**File:** `src/algorithms/baseline/knn_base.py`  
**Classes:** `KNNBase`, `KNNClassifier`, `KNNClassifierFast`

All KNN variants in this project share a common class hierarchy:

| Class | Role |
|---|---|
| `KNNBase` | Abstract base — stores training data, computes neighbour distances, delegates `aggregate()` |
| `KNNClassifier` | Standard majority vote via `Counter.most_common(1)` |
| `KNNClassifierFast` | Same predictions; replaces the Python distance loop with a single `scipy.cdist` call |

`KNNClassifierFast` is bit-for-bit equivalent to `KNNClassifier` but significantly faster. It is the practical base for all variants. The standard decision rule is:

$$\hat{y}(x) = \arg\max_{c} \sum_{i=1}^{k} \mathbf{1}[y_{(i)} = c], \qquad P(y = c \mid x) = \frac{1}{k} \sum_{i=1}^{k} \mathbf{1}[y_{(i)} = c]$$

where $y_{(i)}$ is the label of the $i$-th nearest neighbour of query $x$. `KNNClassifier` with a fixed $k$ is not benchmarked directly — `KNNOptK` is the standard-KNN representative, since selecting $k$ by cross-validation is the appropriate baseline.

---

<a id="knnoptk"></a>

### 1.1 KNNOptK — KNN with Cross-Validated k

**File:** `src/algorithms/baseline/knn_base.py`  
**Class:** `KNNOptK`

---
#### Theory

Standard KNN requires choosing $k$ before seeing the data: a small $k$ is noisy (high variance), a large $k$ smears the decision boundary (high bias). The optimal $k$ depends on the dataset size $N$ and the local class geometry, so no single fixed value works across datasets.

KNNOptK frames $k$-selection as a model-selection problem: use the training data itself to estimate which $k$ performs best via **inner stratified cross-validation**, re-derived from scratch on each training fold of the outer CV.

---
#### How it works

1. Build a candidate grid of odd $k$ values from the training set of size $N$:
   $$\mathcal{K} = \{1, 3, 5, \ldots, \lfloor\sqrt{N}\rfloor\}$$
   Odd values only — avoids tie votes in binary classification. The $\sqrt{N}$ upper bound scales automatically with dataset size.

2. Run stratified $K$-fold cross-validation on the training set. For every combination $(k, \text{fold})$, fit `KNNClassifierFast(k)` on the fold's training portion and evaluate **balanced accuracy** on the held-out portion:
   $$\text{BalAcc}(k, \text{fold}) = \frac{1}{2}\left(\frac{TP}{P} + \frac{TN}{N}\right)$$
   All $(k, \text{fold})$ pairs are evaluated in parallel via `joblib`.

   Balanced accuracy is used instead of plain accuracy because under imbalance a trivial majority-class predictor achieves high accuracy but $\text{BalAcc} = 0.5$.

3. Average balanced accuracy across folds for each $k$ and select the winner:
   $$k^* = \arg\max_{k \in \mathcal{K}} \; \frac{1}{K} \sum_{\text{fold}} \text{BalAcc}(k, \text{fold})$$

4. Fit a final `KNNClassifierFast(k=k^*)` on the **full training set**.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_max` | $\lfloor\sqrt{N}\rfloor$ | Upper bound on $k$ candidates |
| `inner_cv_folds` | config | Number of inner CV folds |
| `n_jobs` | 8 | Parallel workers for $(k, \text{fold})$ evaluation |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** (inner CV) | $O(\lvert\mathcal{K}\rvert \cdot N^2 d) = O(N^{5/2} d)$ |
| **Predict** | $O(N_{\text{test}} \cdot N \cdot d)$ |

The inner CV dominates: $\lvert\mathcal{K}\rvert \approx \sqrt{N}/2$ candidate values, each evaluated across $K$ folds. For each fold the `cdist` call between the test portion ($\approx N/K$ points) and the training portion ($\approx N(1-1/K)$ points) costs $O(N^2 d / K)$. Summed over folds and candidates: $O(\sqrt{N}/2 \cdot N^2 d)$. The final fit and all predictions are single `cdist` calls.

---
#### Limitations & what it motivated

KNNOptK exposes the ceiling of fixing bias through $k$-selection alone. Inner CV ends up choosing $k = 1$ in roughly 62% of folds across the benchmark, confirming that the optimal global response to imbalance is *maximal locality*. But even at $k = 1$, performance still degrades as IR grows — choosing fewer neighbours does not remove the bias, it only softens it.

The root cause is not the value of $k$ but how the two classes are compared once the neighbours are found. Standard KNN asks whether the *closest minority* neighbour is closer than the *closest majority* neighbour. With $N_{\text{maj}} \gg N_{\text{min}}$, the majority class has more points in every direction, so its closest representative is statistically expected to win that comparison regardless of the true geometry. This is a sampling artefact that persists no matter how small $k$ is.

This observation reframed the whole problem and motivated the **FairRank family**: instead of tuning $k$, correct the *rank* at which the two classes are compared.

---

<a id="knnweighted"></a>

### 1.2 KNNWeighted — Class-Frequency Weighting

**File:** `src/algorithms/baseline/knn_weighted.py`  
**Class:** `KNNWeighted`

---
#### Theory

Standard KNN treats every neighbour's vote equally, implicitly placing a prior proportional to class size. Under imbalance, majority points dominate the vote simply by their abundance.

KNNWeighted corrects this by multiplying each class's vote total by a weight inversely proportional to its training frequency — equivalent to applying a uniform class prior so each class contributes equally regardless of size.

The weight for class $c$ is:
$$w_c = \frac{N_{\text{total}}}{n_{\text{classes}} \cdot N_c}$$

In the binary case this gives $w_{\text{min}} = r$ and $w_{\text{maj}} = 1/r$. The weighted decision rule is:
$$\hat{y}(x) = \arg\max_c \; w_c \cdot \sum_{i=1}^{k} \mathbf{1}[y_{(i)} = c]$$

The normalised probability estimate returned by `predict_proba` is:
$$P(\text{minority} \mid x) = \frac{w_{\text{min}} \cdot \text{count}_{\text{min}}}{w_{\text{min}} \cdot \text{count}_{\text{min}} + w_{\text{maj}} \cdot \text{count}_{\text{maj}}}$$

---
#### How it works

**Concrete example** — $N_{\text{maj}} = 950$, $N_{\text{min}} = 50$ (IR = 19), $k = 5$, query has 4 majority neighbours and 1 minority neighbour:

$$w_{\text{min}} = \frac{1000}{2 \times 50} = 10, \quad w_{\text{maj}} = \frac{1000}{2 \times 950} \approx 0.53$$

- Standard KNN: majority wins 4–1 → predicts **majority**.
- KNNWeighted: majority weighted score $= 4 \times 0.53 = 2.1$; minority score $= 1 \times 10 = 10$ → predicts **minority**.

The weight ratio $w_{\text{min}} / w_{\text{maj}} = r^2$, so a single minority neighbour out-votes up to $r^2$ majority neighbours. This is an aggressive, fixed correction: for IR = 19 a single minority neighbour beats a majority neighbourhood of up to 361 points.

**At fit time:** compute `_class_weights` from training label counts. No neighbourhood computation.  
**At predict time:** retrieve the $k$ nearest neighbours via `KNNClassifierFast`, then apply the weights when aggregating votes. Distance computation is inherited unchanged.

---
#### Hyperparameters

Inherits all parameters from `KNNClassifierFast`. Weights are derived automatically from the training labels at fit time and are not user-configurable.

| Parameter | Default | Meaning |
|---|---|---|
| `k` | 5 | Number of neighbours |
| `distance_func` | Euclidean | Distance metric |
| `n_jobs` | 1 | Parallelism |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** | $O(N)$ — weight computation from class counts |
| **Predict** | $O(N_{\text{test}} \cdot N \cdot d)$ — single `cdist` call, same as `KNNClassifierFast` |

No inner CV. The fit cost is negligible; the predict cost is identical to base KNN.

---
#### Limitations & what it motivated

The weight ratio $w_{\text{min}} / w_{\text{maj}} = r^2$ is a global constant — it does not adapt to where in feature space the query lands. In majority-dense regions, applying the full amplification is too aggressive: a single minority neighbour outweighs hundreds of majority ones even when the query is geometrically far from any minority training point. This generates a systematic excess of false positives at high IR.

The stress-test results confirm this: KNNWeighted wins on datasets in the mild-imbalance quartile (Q1) but degrades to 0 wins in the severe-imbalance quartiles (Q3–Q4). Its one advantage is that it requires no inner CV, making it uniquely robust in the degenerate regime where there are too few minority samples per fold to run a reliable cross-validation.

KNNWeighted serves as the **sanity-check baseline** for the FairRank family: any rank-correction variant that cannot beat it is not adding value beyond simple frequency reweighting.

---

<a id="smote-knn"></a>

### 1.3 SMOTE+KNN — Synthetic Minority Oversampling + KNN

**Library:** `imblearn.over_sampling.SMOTE` + `KNNClassifierFast`

---
#### Theory

SMOTE (Synthetic Minority Over-sampling Technique, Chawla et al. 2002) addresses class imbalance at the **data level** rather than the decision level. Instead of reweighting votes or correcting the comparison rank, it artificially generates new minority training samples until the classes are balanced, then trains a standard classifier on the augmented dataset.

The generation mechanism is based on the local geometry of the minority class. For each existing minority point $x_i$:

1. Find its $k_s$ nearest **minority** neighbours.
2. Randomly select one, $x_j$.
3. Generate a new synthetic point on the line segment between them:
   $$x_{\text{new}} = x_i + \lambda \cdot (x_j - x_i), \quad \lambda \sim \text{Uniform}(0, 1)$$

This creates new examples that interpolate between existing minority samples — they lie in the same sub-region of feature space but are not copies. Synthesis is repeated until the desired class ratio is reached (default: 1:1 balance).

**Key assumption:** minority samples form coherent clusters in feature space and linear interpolation between nearby points stays within the minority distribution. This breaks down when the minority class is highly dispersed or intrinsically non-convex.

---
#### How it works

1. **SMOTE oversampling (pre-processing):** Compute pairwise distances among all $N_{\text{min}}$ minority points, select $k_s$ nearest minority neighbours for each point, draw synthetic samples along random connecting segments. Repeat until $N_{\text{min,new}} = N_{\text{maj}}$.

2. **KNN fit on augmented data:** Fit `KNNClassifierFast` on the merged set: original majority points + original minority points + synthetic minority points. The augmented training set has size $\approx 2 N_{\text{maj}}$.

3. **Predict:** Standard majority vote KNN on the augmented training set — no special inference logic.

After oversampling, the classifier sees a balanced dataset and uses a uniform vote. The imbalance correction is entirely in the data generation step.

---
#### Hyperparameters

| Parameter | Source | Default | Meaning |
|---|---|---|---|
| `k_neighbors` | SMOTE | 5 | Minority neighbours used for synthesis |
| `sampling_strategy` | SMOTE | `'auto'` | Target minority/majority ratio after oversampling |
| `k` | KNNClassifierFast | 5 | Neighbours used at predict time |
| `random_state` | SMOTE | config | Seed for reproducibility |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **SMOTE synthesis** | $O(N_{\text{min}}^2 \cdot d)$ — pairwise distances within the minority class |
| **KNN fit on augmented set** | $O(N_{\text{aug}})$ — lazy storage, where $N_{\text{aug}} \approx 2 N_{\text{maj}}$ |
| **Predict** | $O(N_{\text{test}} \cdot N_{\text{aug}} \cdot d)$ — `cdist` against the augmented training set |

At high imbalance ratios, $N_{\text{aug}} \gg N$, so predict time and storage both grow significantly compared to non-oversampling methods.

---
#### Limitations & what it motivated

SMOTE's interpolation assumption — that minority samples form coherent clusters and that linear paths between nearby minority points stay within the minority distribution — breaks down when the minority class is geometrically dispersed or intrinsically non-convex. In those cases, synthetic points land in majority-dominated regions and actively hurt the classifier.

A second structural problem is scalability: at high IR the augmented training set grows to $\approx 2 N_{\text{maj}}$ points, making predict time significantly slower than any non-oversampling method.

More fundamentally, SMOTE does not address the root cause of KNN's bias — the unfair rank comparison across classes. It corrects for imbalance by changing the data, not by changing how the decision is made. Both framing and data-level approaches have merit; SMOTE+KNN remains in the benchmark as the representative of the data-augmentation family and as a competitive external reference for the rank-correction variants.

---

<a id="fairrank-family"></a>

## 2. FairRank Family

This is the main contribution of the project. All variants share the same theoretical foundation.

### 2.0 Why Standard KNN Is Structurally Biased

Imagine two classes with **identical spatial density** — same shape, same spread — but very different sizes: $N_{maj} = 1000$, $N_{min} = 50$ (IR = 20).

Pick a query $x$ sitting right on the true decision boundary. Even though both classes are equally dense there, the nearest majority neighbour is almost certainly closer than the nearest minority neighbour. Why? Simply because with 1000 majority points there are more of them in every direction, and random sampling guarantees that some land very close to $x$ by chance.

**This is not a signal about the data. It is pure sampling noise from having more points.**

Formally, under a homogeneous Poisson process model the expected distance to the $k$-th nearest class-$c$ neighbour scales as:
$$E[d_k^c] \propto \left(\frac{k}{N_c}\right)^{1/d}$$

We want to find the majority rank $k_{eff}$ such that the $k_{eff}$-th majority neighbour is at the **same expected distance** as the 1st minority neighbour:
$$E[d_{k_{eff}}^{maj}] = E[d_1^{min}]$$
$$\left(\frac{k_{eff}}{N_{maj}}\right)^{1/d} = \left(\frac{1}{N_{min}}\right)^{1/d}$$

The dimension $d$ cancels on both sides:
$$\frac{k_{eff}}{N_{maj}} = \frac{1}{N_{min}} \implies \boxed{k_{eff} = \frac{N_{maj}}{N_{min}} = r}$$

**The fair rank is dimension-free** — it does not depend on how many features the data has.

**Intuition:** With IR = 20, you need to look at the 20th majority neighbour to find one that is at the same statistically expected distance as the 1st minority neighbour. The 1st majority neighbour is not a fair comparison — it wins by sheer numbers, not by proximity signal.

---

<a id="knnfairrank"></a>

### 2.1 KNNFairRank — Core Algorithm

**File:** `src/algorithms/fair_rank/core/knn_fair_rank.py`  
**Class:** `KNNFairRank`

---
#### Theory

The derivation in §2.0 gives the correction for a single rank comparison: compare the 1st minority neighbour against the $r$-th majority neighbour. KNNFairRank (v3) extends this with **multi-rank voting** to reduce per-query variance: instead of a single binary decision, run $n_{votes}$ independent rank-corrected comparisons and take the majority vote.

For comparison $i$, the fair pairing is the $i$-th minority rank vs. the $(i \cdot r)$-th majority rank — the same Poisson scaling argument applies at every rank, so the correction holds for all $i$ simultaneously.

Separate per-class distance lists replace the single joint neighbourhood to ensure minority points are always represented regardless of imbalance ratio.

---
#### How it works

**At fit time:**
1. Split the training set: store $X_{\text{min}}$ (all minority points) and $X_{\text{maj}}$ (all majority points) separately.
2. Compute $r = N_{\text{maj}} / N_{\text{min}}$ and set $k_{\text{eff}} = \text{round}(r)$.
3. Size the majority neighbourhood to cover $n_{\text{votes}} \times k_{\text{eff}}$ positions with a safety margin:
   $$k_{\text{maj}} = \text{clip}\!\left(\lceil r \rceil \cdot n_{\text{votes}} + \text{buffer},\; k_{\text{floor}},\; k_{\text{cap}}\right)$$

**At predict time — for a query $x$:**

1. Compute sorted distances to all minority points: $d_1^{\text{min}} \leq d_2^{\text{min}} \leq \cdots$
2. Compute sorted distances to all majority points: $d_1^{\text{maj}} \leq d_2^{\text{maj}} \leq \cdots$
3. Run $n_{\text{votes}}$ rank-corrected comparisons. For comparison $i$:
   - Minority side: $d_i^{\text{min}}$
   - Majority side: $d_{i \cdot k_{\text{eff}}}^{\text{maj}}$
   - Vote minority if $d_i^{\text{min}} < d_{i \cdot k_{\text{eff}}}^{\text{maj}}$, else vote majority.
4. Predict **minority** if the minority vote fraction $> 0.5$, else predict **majority**.

**Concrete example** — IR = 5, $k_{\text{eff}} = 5$, $n_{\text{votes}} = 3$:

| Comparison $i$ | Minority dist $d_i^{\text{min}}$ | Majority dist $d_{5i}^{\text{maj}}$ | Winner |
|:---:|:---:|:---:|:---:|
| 1 | 0.42 | 0.61 | minority ✓ |
| 2 | 0.55 | 0.50 | majority ✗ |
| 3 | 0.70 | 0.88 | minority ✓ |

Vote fraction = 2/3 > 0.5 → **predict minority**.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_min` | 10 | Number of minority neighbours fetched per query |
| `k_maj_buffer` | 10 | Extra majority neighbours beyond the theoretical minimum |
| `k_maj_floor` | 30 | Minimum majority neighbourhood size (stability at low IR) |
| `k_maj_cap` | 1000 | Cap on majority neighbourhood size (prevents memory blowup at extreme IR) |
| `n_votes` | 5 | Number of rank-corrected comparisons per query |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** | $O(N)$ — split into class arrays, compute $r$ |
| **Predict** | $O(N_{\text{test}} \cdot (N d + N \log N))$ — two `cdist` calls + sort, then $O(n_{\text{votes}})$ voting |

For large $d$, the `cdist` dominates: effectively $O(N_{\text{test}} \cdot N \cdot d)$. The sort costs $O(N \log N)$ per query and is negligible in high dimensions.

---
#### Limitations & what it motivated

KNNFairRank (v3) derives $k_{\text{eff}} = r$ under a key assumption: both classes have **uniform and equal local density** everywhere. When that assumption holds, the correction is exact. When it breaks — minority clustered in sub-regions, majority spread unevenly — the global $r$ over-corrects in majority-dense regions, predicting minority when the query is genuinely in majority territory. This manifests as a high false-positive rate: excellent G-Mean (highest minority recall in the benchmark) but last place on MCC.

Two distinct failure modes were identified:
1. **Density non-uniformity**: the true local imbalance ratio varies across the feature space, but the correction applies the global $r$ everywhere.
2. **Per-query variance**: a single anomalously close minority training point (duplicate, noise, or outlier) can tip the vote fraction to 1.0 with false confidence.

The first failure mode motivated **KNNFairRankCV** (make the correction strength dataset-adaptive via $\alpha$) and eventually **KNNFairRankTopoJointBootstrap** (make it region-adaptive). The second motivated **KNNFairRankJackknife** (average out the influence of individual outlier points at inference time).

---

<a id="knnfairrankcv"></a>

### 2.2 KNNFairRankCV — Cross-Validated Correction Strength

**File:** `src/algorithms/fair_rank/core/knn_fair_rank_c.py`  
**Class:** `KNNFairRankCV`

---
#### Theory

The base algorithm sets $k_{\text{eff}} = r$ under the assumption of uniform density. When that assumption breaks — minority clustered in sub-regions, majority spread unevenly — $k_{\text{eff}} = r$ over-corrects: in majority-dense regions the full correction is too aggressive, generating false positives.

KNNFairRankCV introduces a single exponent $\alpha \in (0, 1]$ that dials the correction strength:
$$k_{\text{eff}} = \text{round}(r^\alpha)$$

| $\alpha$ | $k_{\text{eff}}$ (IR = 20) | Effect |
|:---:|:---:|---|
| 1.0 | 20 | Full correction — maximum minority sensitivity |
| 0.75 | 9 | Moderate correction |
| 0.5 | $\approx 4$ | Square-root damping |
| 0.25 | 2 | Mild correction |

$\alpha$ is not fixed — it is selected separately for each dataset by inner stratified cross-validation, adapting the correction strength to the actual density structure of that dataset.

---
#### How it works

1. Split the training data into inner CV folds (default: 3-fold stratified).
2. For each candidate $\alpha \in \{0.25, 0.5, 0.75, 1.0\}$ and each fold, fit `KNNFairRank` on the fold's training portion with $k_{\text{eff}}$ scaled by $\alpha$, and score G-Mean on the held-out portion.
3. Average G-Mean across folds for each $\alpha$ and pick the winner:
   $$\alpha^* = \arg\max_{\alpha} \; \overline{\text{G-Mean}}(\alpha)$$
4. Fit the final `KNNFairRank` on the full training set and apply $\alpha^*$ at predict time via:
   $$\text{effective } k_{\text{eff}} = \text{round}(r^{\alpha^*})$$

**Why this fixes over-correction:** on a dataset with uniform density the CV selects $\alpha^* \approx 1.0$. On a dataset where minority is clustered and the global $r$ overstates local imbalance, the CV selects $\alpha^* < 1.0$, shrinking $k_{\text{eff}}$ and reducing false positives. The correction strength is matched to the actual dataset structure.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `alpha_grid` | [0.25, 0.5, 0.75, 1.0] | Candidate correction exponents |
| `inner_cv_folds` | 3 | Folds for the inner CV |
| `scoring` | `geometric_mean` | Metric optimised to pick $\alpha$ |
| Plus all `KNNFairRank` parameters | — | — |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** (inner CV) | $O(\lvert\alpha_{\text{grid}}\rvert \cdot N^2 d) = O(4 N^2 d)$ |
| **Predict** | $O(N_{\text{test}} \cdot N \cdot d)$ |

With 4 candidates and 3 folds, 12 inner `KNNFairRank.fit()` calls are made. The predict cost is identical to base `KNNFairRank`.

---
#### Limitations & what it motivated

KNNFairRankCV selects $\alpha$ by inner CV while keeping $n_{\text{votes}}$ fixed at its default. But the two hyperparameters are not independent: the optimal $\alpha$ depends on $n_{\text{votes}}$ and vice versa. Optimising one while holding the other fixed navigates only a one-dimensional slice of a two-dimensional space and may miss the true joint optimum.

A secondary limitation is that the inner CV still applies a single $\alpha$ across the entire feature space. It adapts the correction to the dataset as a whole, but not to local density variation within the dataset. A region with very high local minority density still gets the same $k_{\text{eff}}$ as a region where minority is sparse.

The first limitation directly motivated **KNNFairRankJointCV** (search the full $(\alpha, n_{\text{votes}})$ grid jointly). The second was the longer-term motivation for the topology-based variants.

---

<a id="knnfairrankjointcv"></a>

### 2.3 KNNFairRankJointCV — Joint Cross-Validation of $\alpha$ and $n_{\text{votes}}$

**File:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_joint_cv.py`  
**Class:** `KNNFairRankJointCV`

---
#### Theory

`KNNFairRankCV` (§2.2) tunes $\alpha$ while keeping $n_{\text{votes}}$ fixed at its default. But $\alpha$ and $n_{\text{votes}}$ interact: a high $n_{\text{votes}}$ with low $\alpha$ produces many conservative comparisons (much evidence, each comparison under-corrected), while a low $n_{\text{votes}}$ with high $\alpha$ produces few aggressive comparisons (less evidence, each fully corrected). Optimising one while fixing the other finds a local optimum that misses the interaction.

KNNFairRankJointCV searches the **full Cartesian grid** jointly:

$$\text{Grid: } (n_{\text{votes}}, \alpha) \in \{1, 2, 3, 5, 7, 10\} \times \{0.25, 0.5, 0.75, 1.0\} \quad (24 \text{ pairs})$$

---
#### How it works

1. Run stratified $K$-fold CV on the training data. For every pair $(n_v, \alpha)$ in the grid, fit `KNNFairRank` on the fold's training portion with $k_{\text{eff}} = \text{round}(r^\alpha)$ and $n_{\text{votes}} = n_v$, and score G-Mean on the held-out portion.
2. Average G-Mean across folds for each pair and select the winner:
   $$(n_v^*, \alpha^*) = \arg\max_{(n_v, \alpha)} \; \overline{\text{G-Mean}}(n_v, \alpha)$$
3. Fit the final `KNNFairRank` on the full training set with $n_{\text{votes}} = n_v^*$ and $\alpha^*$ applied at predict time.

**Why joint search beats separate tuning:** suppose $\alpha = 1.0$ wins when tested with the default $n_{\text{votes}} = 5$. But with $n_{\text{votes}} = 3$, $\alpha = 0.75$ might score higher because a tighter correction over fewer votes better matches that dataset's density profile. The joint search discovers these interactions; `KNNFairRankCV` alone cannot.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `alpha_grid` | [0.25, 0.5, 0.75, 1.0] | Candidate correction exponents |
| `n_votes_grid` | [1, 2, 3, 5, 7, 10] | Candidate vote counts |
| `inner_cv_folds` | config | Number of inner CV folds |
| `scoring` | `geometric_mean` | Metric optimised to pick $(n_v^*, \alpha^*)$ |
| Plus all `KNNFairRank` parameters | — | — |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** (inner CV) | $O(\lvert\alpha_{\text{grid}}\rvert \cdot \lvert n_{\text{votes grid}}\rvert \cdot N^2 d) = O(24 N^2 d)$ |
| **Predict** | $O(N_{\text{test}} \cdot N \cdot d)$ |

With 24 pairs and 3 folds, 72 inner `KNNFairRank.fit()` calls are made — 6× the cost of `KNNFairRankCV`. The predict cost is identical to base `KNNFairRank`.

---
#### Limitations & what it motivated

KNNFairRankJointCV finds the best global $(\alpha^*, n_v^*)$ for the whole dataset and applies it uniformly to every query. The joint search is more complete than separate tuning, but the result is still a dataset-level constant — it cannot adapt to the fact that different regions of the feature space may need different correction strengths.

The inner CV over 24 pairs is also the most expensive fit procedure in the benchmark. On datasets with very few minority samples per fold (the degenerate regime), the 24-pair grid cannot be scored reliably and the algorithm drops to 4th place, where KNNWeighted — which requires no optimisation at all — outperforms it.

These limitations motivated two further directions: **KNNFairRankTopoJointBootstrap** (replace the global $k_{\text{eff}}$ with a per-region one, derived from Ward clustering on the joint point cloud) and **KNNFairRankOptVotes** (motivated by observing that the optimal $n_{\text{votes}}$ follows a strongly bimodal distribution across datasets, meaning the joint grid often wastes evaluations in the dead zone).

---

<a id="knnfairrankjackknife"></a>

### 2.4 KNNFairRankJackknife — Leave-One-Out Minority Robustness

**File:** `src/algorithms/fair_rank/resampling/knn_fair_rank_jackknife.py`  
**Class:** `KNNFairRankJackknife`

---
#### Theory

Base KNNFairRank has a fragility: if a single minority training point happens to land very close to a query $x$ — due to a duplicate, a mislabelled sample, or random sampling noise — it wins comparison $i=1$ by a large margin and can tip the vote toward minority regardless of the other comparisons. This inflates the vote fraction and overestimates minority probability.

KNNFairRankJackknife makes the per-query decision robust to this by running **Leave-One-Out trials over the minority distance sequence**. For each trial, one minority point is removed and the vote fraction is recomputed. The final prediction averages across all trials.

The intuition is jackknife variance estimation applied at inference time: if a single minority point dominates the decision, its removal will substantially change the vote fraction. Averaging tempers that influence while preserving the signal from the genuine neighbourhood structure.

---
#### How it works

For a query $x$ with sorted minority distances $[d_1^{\text{min}}, \ldots, d_{k_{\text{min}}}^{\text{min}}]$:

1. For each trial $j = 1, \ldots, k_{\text{probe}}$:
   - Remove $d_j^{\text{min}}$ from the minority list — pretend that training point does not exist.
   - Run the standard v3 vote step on the remaining $k_{\text{min}} - 1$ distances (ranks shift down by one after the removal).
   - Record the resulting vote fraction $f_j$.

2. Average across all trials:
   $$\bar{f} = \frac{1}{k_{\text{probe}}} \sum_{j=1}^{k_{\text{probe}}} f_j$$

3. Predict minority if $\bar{f} > 0.5$.

**Concrete illustration** — $n_{\text{votes}} = 3$, $k_{\text{probe}} = 3$, with a noisy minority point at rank 1:

| Trial $j$ | Point removed | Vote fraction $f_j$ |
|:---:|:---:|:---:|
| 1 | $d_1^{\text{min}}$ (the outlier) | 0.33 |
| 2 | $d_2^{\text{min}}$ | 0.67 |
| 3 | $d_3^{\text{min}}$ | 0.67 |

$\bar{f} = 0.56$ — still predicts minority, but with lower confidence. If the noisy point had produced $f = 1.0$ in base v3, the jackknife average tempers it to 0.56, giving a better-calibrated soft score.

**Constraint on `k_probe`:** after removing one point, at least $n_{\text{votes}}$ minority distances must remain. So `k_probe` is automatically capped at `k_min - n_votes`. On degenerate datasets with too few minority samples, the algorithm falls back to standard v3.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `k_probe` | `n_votes` | Number of LOO trials (auto-capped at `k_min - n_votes`) |
| Plus all `KNNFairRank` parameters | — | — |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** | $O(N)$ — lazy storage, same as base `KNNFairRank` |
| **Predict** | $O(N_{\text{test}} \cdot (N d + k_{\text{probe}} \cdot n_{\text{votes}}))$ |

The `cdist` calls are identical to base KNNFairRank. The extra cost is $k_{\text{probe}}$ vote re-evaluations per query — each re-evaluation costs $O(n_{\text{votes}})$ index operations on already-computed distance arrays. With default $k_{\text{probe}} = n_{\text{votes}} = 5$, this is 25 extra comparisons per query — negligible relative to the $O(N d)$ distance computation.

---
#### Limitations & what it motivated

KNNFairRankJackknife addresses a specific failure mode — an individual outlier minority point dominating the vote — but it does not help when the failure is systematic. If $k_{\text{eff}}$ is globally wrong (because the density assumption is violated everywhere), jackknife averaging will not recover the decision: all $k_{\text{probe}}$ trials will be wrong in the same direction, and the average will faithfully reproduce the error.

The per-query overhead of $k_{\text{probe}}$ vote re-evaluations is negligible in practice, but the approach also reaches a natural ceiling at $k_{\text{probe}} = k_{\text{min}} - n_{\text{votes}}$: there is a limit to how many minority points can be left out while still running a valid vote step. On degenerate datasets with very few minority training samples per fold, the jackknife collapses to standard v3.

Within the benchmark, KNNFairRankJackknife functions as a **calibration complement** to KNNFairRankCV: it is best at reducing overconfident soft scores (strong ROC-AUC, 2nd overall) while the CV variants are better at reducing systematic false positives.

---

<a id="knnfairrankoptvotes"></a>

### 2.5 KNNFairRankOptVotes — Cross-Validated Vote Count

**File:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_opt_votes.py`  
**Class:** `KNNFairRankOptVotes`

---
#### Theory

Base KNNFairRank uses a fixed $n_{\text{votes}}$. The optimal vote count is not constant across datasets — empirical analysis across 45 datasets reveals a strongly bimodal distribution of optimal values:

- 28/45 datasets peak at $n_{\text{votes}} \in \{1, 2\}$ — *few-votes* regime: a single clean comparison suffices, additional votes add noise
- 12/45 datasets peak at $n_{\text{votes}} \in \{7, 10\}$ — *many-votes* regime: multiple comparisons are needed to average out per-query variance

The default $n_{\text{votes}} = 5$ is near-optimal for only 7/45 datasets — it sits in the dead zone between the two clusters. No single fixed value covers both regimes.

KNNFairRankOptVotes treats $n_{\text{votes}}$ as a dataset-specific hyperparameter and selects it via inner cross-validation, analogously to how `KNNFairRankCV` selects $\alpha$.

---
#### How it works

**At fit time:**

1. Size the majority neighbourhood for the *largest* candidate in `n_votes_grid` — this ensures all candidates can be evaluated without recomputing distances:
   $$k_{\text{maj}} = \text{clip}\!\left(\lceil r \rceil \cdot n_{\text{votes,max}} + \text{buffer},\; k_{\text{floor}},\; k_{\text{cap}}\right)$$

2. Run stratified $K$-fold CV. For each pair $(n_v, \text{fold})$, fit `KNNFairRank(n_votes=n_v)` on the fold's training portion and score G-Mean on the held-out portion. All pairs are evaluated in parallel.

3. Average G-Mean across folds for each $n_v$ and select:
   $$n_v^* = \arg\max_{n_v \in \text{grid}} \; \overline{\text{G-Mean}}(n_v)$$

4. Set `self._n_votes = n_v^*` and fit the final model on the full training set.

**At predict time:** standard KNNFairRank prediction using $n_{\text{votes}} = n_v^*$.

The inner CV evaluates different vote counts by indexing into already-sorted distance arrays — the expensive `cdist` calls happen once per fold, not once per candidate. The overhead is $|\text{grid}| \times K_{\text{folds}}$ additional `KNNFairRank.fit()` calls.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `n_votes_grid` | [1, 2, 3, 5, 7, 10] | Candidate vote counts |
| `inner_cv_folds` | config (default: 3) | Inner CV folds |
| `scoring` | `geometric_mean` | Metric used to select $n_v^*$ |
| Plus all `KNNFairRank` parameters | — | — |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** (inner CV) | $O(\lvert n_{\text{votes grid}}\rvert \cdot N^2 d) = O(6 N^2 d)$ |
| **Predict** | $O(N_{\text{test}} \cdot N \cdot d)$ |

With 6 candidates and 3 folds, 18 inner fit calls are made — comparable to `KNNFairRankCV` (4 × 3 = 12). The predict cost is identical to base `KNNFairRank`.

---
#### Limitations & what it motivated

KNNFairRankOptVotes selects the best $n_{\text{votes}}$ but keeps $\alpha = 1$ (full rank correction) fixed. It therefore does not address the density over-correction problem — on datasets where the minority is clustered and the global $r$ is too aggressive, selecting a better vote count cannot compensate for a systematically wrong $k_{\text{eff}}$.

The algorithm's main finding — the strongly bimodal distribution of optimal $n_{\text{votes}}$ — is also an implicit argument for the joint CV approach: if the optimal $n_{\text{votes}}$ is bimodal, then fixing $\alpha$ and tuning $n_{\text{votes}}$ independently will still find only a conditional optimum. **KNNFairRankJointCV** subsumes this search by tuning both simultaneously, which is why OptVotes and JointCV are kept as distinct entries rather than one superseding the other: they test different axes of the hyperparameter space.

---

<a id="knnfairrankensemble"></a>

### 2.6 KNNFairRankEnsemble — Full-Grid Vote Pooling

**File:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_ens.py`  
**Class:** `KNNFairRankEnsemble`

---
#### Theory

`KNNFairRankCV` picks one $(\alpha, n_{\text{votes}})$ setting via inner CV and commits to it at predict time. This is a winner-takes-all strategy: the selected pair may perform well on average but can still fail on individual queries where a different pair would be better.

KNNFairRankEnsemble takes the opposite approach: **use the entire grid at prediction time** and pool all votes together. Every $(n_v, \alpha)$ combination contributes its individual votes to a single aggregated vote fraction.

The key insight is that diversity of correction strengths produces soft scores spread across $[0, 1]$ rather than quantised to $\{0, 1/n_v, \ldots, 1\}$. Conservative pairs (low $\alpha$) act as a brake on the aggressive pairs (high $\alpha$), producing better-calibrated probability estimates that directly benefit ROC-AUC and PR-AUC.

---
#### How it works

For a query $x$, and for every combination $(n_v, \alpha) \in \{1,2,3,5,7,10\} \times \{0.25, 0.5, 0.75, 1.0\}$ (24 pairs):
- Set $k_{\text{eff}}(\alpha) = \text{round}(r^\alpha)$
- Run the $n_v$ rank-corrected comparisons and collect all individual binary votes

Pool all individual votes across all 24 pairs:
$$\text{vote fraction} = \frac{\text{total minority wins across all pairs}}{\text{total votes cast across all pairs}}$$

Predict minority if this fraction $> 0.5$.

**Worked example** — IR = 10, partial grid:

| $(n_v, \alpha)$ | $k_{\text{eff}}$ | Votes cast | Minority wins |
|:---:|:---:|:---:|:---:|
| (1, 0.25) | 2 | 1 | 0 |
| (1, 0.5) | 3 | 1 | 1 |
| (1, 1.0) | 10 | 1 | 1 |
| (3, 0.5) | 3 | 3 | 2 |
| (3, 1.0) | 10 | 3 | 2 |
| (5, 1.0) | 10 | 5 | 3 |
| … | … | … | … |
| **Total** | — | **60** | **33** |

Vote fraction = 33/60 = 0.55 → predict minority.

Pairs with more votes ($n_v$ large) contribute proportionally more — naturally downweighting the noisiest single-vote pairs. When all pairs agree (fraction near 0 or 1) the prediction is high-confidence. When they disagree (fraction near 0.5) the borderline case tends to be classified as majority, reducing false positives.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `alpha_grid` | [0.25, 0.5, 0.75, 1.0] | Correction exponent candidates |
| `n_votes_grid` | [1, 2, 3, 5, 7, 10] | Vote count candidates |
| Plus `k_min`, `k_maj_*` from `KNNFairRank` | — | — |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Fit** | $O(N)$ — lazy storage, same as base `KNNFairRank` |
| **Predict** | $O(N_{\text{test}} \cdot (N d + \lvert\alpha_{\text{grid}}\rvert \cdot \lvert n_{\text{votes grid}}\rvert \cdot n_{\text{votes,max}}))$ |

The `cdist` calls dominate for large $N$ or $d$. The voting overhead is $24 \times 10 = 240$ index operations per query — negligible. No inner CV is needed at fit time.

---
#### Limitations & what it motivated

KNNFairRankEnsemble achieves the best ROC-AUC and PR-AUC in the benchmark but ranks 9th on G-Mean. The reason is the conservative $(\alpha, n_v)$ pairs: low-$\alpha$ grid members use a weak correction and vote minority rarely, acting as a constant brake on minority sensitivity. This is beneficial for precision (fewer false positives) but harmful for recall (more false negatives), which is exactly what G-Mean penalises.

The ensemble also requires no inner CV (purely inference-time pooling), which makes it unusually robust on small datasets and in the degenerate regime — but at the cost of always applying all 24 pairs regardless of what the data actually needs. There is no mechanism for the ensemble to learn that, for a particular dataset, all the low-$\alpha$ pairs are noise.

This suggests a natural extension: weight the pairs by their per-dataset cross-validated score rather than pooling them uniformly. That direction was not pursued in this project but remains an open variant.

---

<a id="knnfairranktopo"></a>

### 2.7 KNNFairRankTopoJointBootstrap — Topology-Adaptive Correction with OOB Reliability Gating

**File:** `src/algorithms/fair_rank/topology/knn_fair_rank_topo_joint_bootstrap.py`  
**Class:** `KNNFairRankTopoJointBootstrap`

---
#### Theory

All previous FairRank variants use a single global correction derived from the global imbalance ratio $r = N_{\text{maj}} / N_{\text{min}}$. But the imbalance is rarely uniform: in some regions of the feature space, minority points are nearly as dense as majority points (local IR $\approx 1$), while in others the minority is vanishingly sparse (local IR $\gg r$). A single global $k_{\text{eff}}$ over-corrects in the first case and under-corrects in the second.

**KNNFairRankTopoJoint** (the base class) addresses this by partitioning the training space into regions using Ward hierarchical clustering on the joint point cloud, then assigning each region its own local $k_{\text{eff}}$:

$$k_{\text{eff}}(R_j) = \min\!\left(\frac{N_{\text{maj}}(R_j) + \lambda}{N_{\text{min}}(R_j) + \lambda},\; r\right)$$

where $\lambda$ is a Laplace smoothing term that prevents extreme values in small regions.

**The reliability problem:** topology can be *stably wrong*. If Ward finds the same incorrect partition every bootstrap resample, the stability of the partition is high even when the per-region correction consistently hurts minority recall. Stability alone cannot detect this failure mode.

**KNNFairRankTopoJointBootstrap** adds **Out-Of-Bag (OOB) reliability gating**: directly measure whether the topology correction produces *better predictions* than a stable reference correction on held-out training points, and fall back to `KNNFairRankJointCV` for any training region where topology is unreliable.

---
#### How it works

**At fit time:**

1. Fit `KNNFairRankTopoJoint` on the full training fold to obtain the Ward partition and per-region $k_{\text{eff}}$ values.

2. Generate $B$ stratified bootstrap resamples. For each bootstrap $b$, approximately 37% of training points are left out (OOB). For each OOB set:
   - Fit a bootstrap `KNNFairRankTopoJoint` on the bootstrap sample.
   - Predict each OOB point using two rules:
     - **Topology prediction:** use the bootstrap model's per-region $k_{\text{eff}}$ for that point's region.
     - **Baseline prediction:** use a fixed $k_{\text{eff}} = \text{round}(r^{\alpha_{\text{baseline}}})$ (a stable JointCV-like reference, default $\alpha_{\text{baseline}} = 0.75$).
   - Compute a differential score per OOB point $i$:
     $$s_b(i) = \mathbf{1}[\text{topo correct}] - \mathbf{1}[\text{baseline correct}] \in \{-1, 0, +1\}$$

3. Aggregate OOB scores across bootstraps for each training point:
   $$\text{reliability}(i) = \frac{1}{|\{b : i \in \text{OOB}_b\}|} \sum_{b : i \in \text{OOB}_b} s_b(i)$$
   Positive → topology consistently outperforms baseline near this point; negative → baseline wins.

4. If any training point has negative reliability, fit a `KNNFairRankJointCV` as the fallback model.

**At predict time:** assign query $x$ to the region of its nearest training point $i^*$. If $\text{reliability}(i^*) \geq 0$, use the topology per-region $k_{\text{eff}}$; otherwise fall back to `KNNFairRankJointCV` predictions.

---
#### Hyperparameters

| Parameter | Default | Meaning |
|---|---|---|
| `n_bootstrap` | 20 | Bootstrap resamples for OOB reliability estimation |
| `oob_baseline_alpha` | 0.75 | Exponent for the OOB comparison baseline: $k_{\text{eff}} = \text{round}(r^{\alpha})$ |
| `fallback_alpha` | None | Pre-computed JointCV $\alpha$ (skips internal JointCV fit if provided) |
| `fallback_n_votes` | None | Pre-computed JointCV $n_{\text{votes}}$ |
| `min_persistence_ratio` | 0.05 | Minimum Ward gap/diameter ratio for accepting a cluster split |
| `laplace_smooth` | 1.0 | $\lambda$ in per-region $k_{\text{eff}}$ formula |
| `min_region_samples` | adaptive | Minimum training points per Ward region |
| Plus all `KNNFairRank` parameters | — | — |

---
#### Time complexity

| Phase | Cost |
|---|---|
| **Base TopoJoint fit** | $O(N^2 d)$ pairwise distances + $O(N^2 \log N)$ Ward linkage |
| **OOB bootstrap evaluation** | $O(B \cdot N^2 d / B) = O(N^2 d)$ — bootstraps run in parallel |
| **JointCV fallback fit** (if needed) | $O(24 \cdot N^2 d)$ — see §2.3 |
| **Predict** | $O(N_{\text{test}} \cdot N \cdot d)$ — `cdist` + nearest-point region lookup |

The Ward clustering step $O(N^2 \log N)$ dominates the fit for large $N$. This is the most expensive algorithm in the benchmark at fit time.

---
#### Limitations & what it motivated

Ward hierarchical clustering operates on the **joint point cloud** of all training points. At high IR, the joint cloud is heavily majority-dominated, so the Ward partition tends to reflect majority class structure rather than the boundary between classes. Regions that are genuinely minority-dense may be grouped together with surrounding majority points if the joint density is smooth, causing the local $k_{\text{eff}}$ to underestimate how minority-rich that region actually is.

The OOB gating mitigates this: if the per-region correction is unreliable (negative OOB reliability score), the algorithm falls back to KNNFairRankJointCV for that query. But the fallback is coarse — it is triggered by the nearest training point's reliability score, not by any direct per-query assessment of whether topology is trustworthy at exactly that location.

This is also the most expensive algorithm in the benchmark at fit time: Ward linkage costs $O(N^2 \log N)$, and the OOB bootstrap adds $B$ additional topology fits on top. On large datasets the fit time can be prohibitive.

As the final variant in the development sequence, KNNFairRankTopoJointBootstrap has no direct successor in the benchmark. The remaining open directions — Bayesian $\alpha$ shrinkage blending global CV with local density estimates, and using shared intrinsic dimensionality to improve per-region $k_{\text{eff}}$ estimation — are documented in the exploration notebooks but were not evaluated in the final benchmark.

---